In [0]:
import requests
import json
from pyspark.sql import functions as F

API_KEY = "MEGPLOTQVE49UBAW"

quote_tickers = [
    "IBM",
    "MSFT",
    "AAPL",
    "BA"
]

def get_quote_json(symbol):
    url = (
        f"https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol={symbol}&outputsize=compact&datatype=json&apikey=MEGPLOTQVE49UBAW"
    )
    response = requests.get(url, timeout=10)
    return response.text

def run_quote_landing_etl(symbol):
    table_name = f"quote_{symbol.lower()}"
    
    # 1. Fetch the data right here on the driver
    raw_json = get_quote_json(symbol)

    # 2. Convert to a standard Spark DataFrame
    df = spark.createDataFrame(
        [(symbol, raw_json)],
        ["symbol", "raw_json"]
    )

    # 3. Add metadata columns
    processed_df = (
        df
        .withColumn("_source", F.lit("Alpha Vantage GLOBAL_QUOTE API"))
        .withColumn("_ingestion_timestamp", F.current_timestamp())
    )

    # 4. Write directly to your Delta/Unity Catalog landing schema
    (processed_df.write
     .format("delta")
     .mode("overwrite") # Swapped to overwrite to match your initial pattern
     .saveAsTable(f"`jarvis-catalog`.landing.{table_name}")
    )

# Run the normal ETL loop
for ticker in quote_tickers:
    run_quote_landing_etl(ticker)

In [0]:
company_tickers = [
    "IBM",
    "MSFT",
    "AAPL",
    "BA"
]

def get_company_json(symbol):
    url = (
        "https://www.alphavantage.co/query"
        f"?function=OVERVIEW"
        f"&symbol={symbol}"
        f"&apikey={API_KEY}"
    )
    response = requests.get(url, timeout=10)
    return response.text

def run_company_landing_etl(symbol):
    table_name = f"company_{symbol.lower()}"

    # 1. Fetch the data directly on the driver node
    raw_json = get_company_json(symbol)

    # 2. Build the local DataFrame
    df = spark.createDataFrame(
        [(symbol, raw_json)],
        ["symbol", "raw_json"]
    )

    # 3. Append metadata columns
    processed_df = (
        df
        .withColumn("_source", F.lit("Alpha Vantage OVERVIEW API"))
        .withColumn("_ingestion_timestamp", F.current_timestamp())
    )

    # 4. Save directly as a standard Delta table in Unity Catalog
    (processed_df.write
     .format("delta")
     .mode("overwrite")
     .saveAsTable(f"`jarvis-catalog`.landing.{table_name}")
    )

# Loop and execute normal ETL processes sequentially
for ticker in company_tickers:
    run_company_landing_etl(ticker)